# 🚛 Chain of Responsibility (CoR) Transport Assistant
## Generative AI Capstone – Integrated Multi-Agent System
### NUS Capstone | July 2025 | Darren Grundy


## 🧠 Project Objective

This capstone demonstrates an integrated AI assistant that coordinates multiple generative AI capabilities, simulating a real-world assistant for transport safety and compliance reporting.

### The system supports:
- 🗣️ Conversational interface with memory
- 📄 Document Question Answering (RAG)
- 🖼️ Text-to-image generation (prompt engineering)
- 🤖 Multi-agent control: Weather | SQL | Recommender
- 📊 Final report on design, debugging, and improvement

**Real-world use case:** An operations assistant managing heavy vehicle Chain of Responsibility (CoR) risks by analyzing incoming telematics data, documents, and compliance reports.


In [2]:
# Optional: mount Google Drive when running in Colab.
# Running locally instead? Skip this and set COR_DATA_DIR in your .env file.
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab - using local COR_DATA_DIR from .env")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
doc_path = str(DATA_DIR / "load_restraint_guide_2025.pdf")


In [4]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Source documents are NOT redistributed in this repository - see the project
# README for official download links, then point COR_DATA_DIR at your copies.
DATA_DIR = Path(os.getenv("COR_DATA_DIR", "./data"))

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "Missing OPENAI_API_KEY. Copy .env.example to .env and add your key."
    )


In [5]:
!pip install -U langchain langchain-community


In [6]:
!pip install pypdf


In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(DATA_DIR / "load_restraint_guide_2025.pdf"))
pages = loader.load_and_split()


In [8]:
!pip install openai python-dotenv langchain chromadb pypdf2 requests pandas matplotlib

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300
)

docs = text_splitter.split_documents(pages)
print(f"Split into {len(docs)} chunks")


Split into 392 chunks


In [10]:
# Install faiss (CPU version for Colab/most environments)
!pip install faiss-cpu


In [11]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# Create embeddings and build FAISS vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)


/tmp/ipython-input-11-2856418136.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [12]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(temperature=0)  # deterministic answers

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


/tmp/ipython-input-12-3039555212.py:4: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0)  # deterministic answers


In [13]:
query = "How should a 10-tonne load be restrained?"
result = qa_chain.run(query)
print(result)


/tmp/ipython-input-13-3786501868.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain.run(query)


A 10-tonne load should be restrained using direct restraint methods such as containing, blocking, or attaching. It is important to ensure that the load is secured properly to prevent movement during all expected conditions of operation, including emergency braking and minor collisions. Additionally, make sure that the load restraint system can withstand the load restraint performance standard forces. If in doubt, it is recommended to get the load restraint system certified by a qualified engineer.


In [14]:
query = "Explain direct vs indirect restraint methods"
result = qa_chain.run(query)
print(result)


Direct restraint methods involve attaching, blocking, or containing a load without relying on friction. This method is particularly useful for loads that are difficult to tie down, such as slippery loads or loads on wheels. Direct lashings, like webbing straps, chains, or twist locks, are used to attach the load directly onto the vehicle. The strength of the direct lashing depends on the weight of the load, the number of lashings, and their direction. The lashing strength is determined by the lashing capacity or manufacturer's rating.

Indirect restraint methods, on the other hand, involve using friction to secure the load. This can include tie-down lashings that rely on the friction between the load and the vehicle deck to prevent movement. Indirect methods like tie-down lashings are suitable for loads where there is enough friction between the load and the vehicle deck. It is important to ensure enough lashings are used with sufficient capacity when using the tie-down restraint metho

In [15]:
query = "Whats the maximum height of a vehicle on Australian roads?"
result = qa_chain.run(query)
print(result)


The maximum height of a loaded vehicle on Australian roads must be safely lower than the height of any obstruction on the journey, such as a bridge or overhead wires. It is essential to ensure that the vehicle's height does not exceed the clearance of any structures along the route to avoid accidents or damage.


In [16]:
query = "Bob has a load in Melbournes CBD - Whats the maximum height of the vehicle allowed?"
result = qa_chain.run(query)
print(result)

The maximum height of a loaded vehicle must be safely lower than the height of any obstruction on the journey, such as a bridge or overhead wires. To determine the exact maximum height allowed for a vehicle in Melbourne's CBD, you would need to know the specific height restrictions for bridges, tunnels, or any other overhead structures along the route. It is recommended to check with the local transport authority or relevant regulatory body for specific height restrictions in Melbourne's CBD.


### 🚧 Observation: Height Restriction Queries
Found that height restrictions were **not being picked up** accurately during document QA.

✅ Action: Add the **RAG height restriction document** (PDF or webpage) to the vector store to enable better handling of:
- Urban load planning (e.g., Melbourne CBD)
- Bridge & overpass clearance
- Load height compliance

This will allow specific location-aware queries (e.g., "What's the max vehicle height in Melbourne CBD?") to return useful results.


In [17]:
doc_path2 = str(DATA_DIR / "HeavyVehiclesHeightClearanceOnRoadsNov2009.pdf")
loader2 = PyPDFLoader(doc_path2)
pages2 = loader2.load_and_split()


In [18]:
docs2 = text_splitter.split_documents(pages2)
print(f"Split height doc into {len(docs2)} chunks")


Split height doc into 12 chunks


In [19]:
all_docs = docs + docs2


In [20]:
vectorstore = FAISS.from_documents(all_docs, embeddings)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


In [21]:
query = "Bob has a load in Melbourne CBD - What’s the maximum height of the vehicle allowed?"
result = qa_chain.run(query)
print(result)


The maximum height allowed for vehicles in Melbourne CBD is up to 4.6 meters according to the information bulletin provided by VicRoads.


### 📄 Document Note: General Dimension Heights in Melbourne CBD

Noted that the height guidance retrieved refers to an **older document (2009)** and mentions a **4.6m limit**, which applies specifically to cattle trucks.

✅ Action: This document was added to the RAG vector store to improve context for:
- Vehicle height compliance
- CBD-specific restrictions
- Differentiating between general freight and livestock conditions

Future queries (e.g., "What's the max height for a general freight truck in Melbourne CBD?") should now return more nuanced answers based on document context.


In [22]:
doc_path3 = str(DATA_DIR / "201602-0113-general-dimension-requirements.pdf")
loader3 = PyPDFLoader(doc_path3)
pages3 = loader3.load_and_split()


In [23]:
docs3 = text_splitter.split_documents(pages3)
print(f"Split general dimension doc into {len(docs3)} chunks")


Split general dimension doc into 9 chunks


In [24]:
all_docs = docs + docs3  # (omit docs2 if that was the older height guide)


In [25]:
vectorstore = FAISS.from_documents(all_docs, embeddings)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


In [26]:
query = "What is the maximum height allowed for a heavy vehicle in Australia?"
response = qa_chain.run(query)
print(response)


The maximum height allowed for a heavy vehicle in Australia is generally 4.3 meters. However, there are exceptions for specific types of vehicles. For example, vehicles built to carry cattle, horses, pigs, or sheep on two decks have a height limit of 4.6 meters. Double-decker buses have a height limit of 4.4 meters.


## 📄 Document-Based QA with RAG: Chain of Responsibility Use Case

This assistant integrates PDF documents related to Australian Chain of Responsibility (CoR) laws, focusing on heavy vehicle compliance. These documents support the assistant’s ability to answer questions using Retrieval-Augmented Generation (RAG).

### 🔍 Documents Included:
1. **Load Restraint Guide 2025** – official guidance on how freight must be restrained to prevent movement.
2. **General Dimension Requirements (2016-02-0113)** – defines maximum legal height, width, and length of heavy vehicles in Australia.

> 🛠️ We removed an outdated 2009 height clearance PDF and replaced it with the 2016 official standard to reflect **current height limits**:
>
> - General maximum height: **4.3m**
> - Livestock and vehicle transporters: **4.6m**
> - Double-decker buses: **4.4m**
>
> This correction ensures our assistant provides legally accurate answers.

### ⚙️ Chunking & Vector Store:
- Text split into ~400–500 chunks using `RecursiveCharacterTextSplitter` with a `chunk_size=1500`, `overlap=300`
- Combined all documents into a FAISS vector store
- Connected to an OpenAI-based retrieval chain via `RetrievalQA`

### ✅ Example Query:
> “What is the maximum height allowed for a heavy vehicle in Australia?”

### 🧠 Answer:
> “The maximum height allowed for a heavy vehicle in Australia is generally **4.3 meters**. However, there are exceptions for specific types of vehicles...”

This confirms accurate document retrieval, context awareness, and structured regulatory insight.


## 👨‍✈️ Driver Database and Telematics Simulation

This prototype simulates how Chain of Responsibility (CoR) obligations can be proactively monitored using AI + telematics + RAG-based document QA.

We define a fleet of **20 drivers** whose telematics data is updated nightly in a batch at **11:59 PM**, enabling next-day compliance checks. Key monitored parameters include:

### 📊 Monitored Parameters:
- **Driver hours** (vs legal driving limits)
- **Speeding events**
- **Mass/dimension breaches** (via load weights and vehicle specs)
- **Rest breaks and fatigue risk**
- **Hazards or incident reports**
- **Maintenance alerts** (linked to fault codes)

### 📡 Telematics-Inferred CoR Responsibilities:
| Area             | Telematics Captures                                | CoR Link                        |
|------------------|-----------------------------------------------------|---------------------------------|
| Load Restraint   | GPS+accel spikes, sudden decel, vibration alerts    | Duty to prevent load shift      |
| Mass Management  | Gross Vehicle Mass (GVM) data from sensors          | Must not exceed 4.5t+ thresholds |
| Dimensions       | Vehicle profile + comparison to declared cargo      | Prevent over-height/over-width  |
| Fatigue          | Continuous driving, breaks, driver ID login/logout | Driver schedules & rest breaks  |
| Maintenance      | Fault codes, alerts, tyre/brake sensor anomalies    | Fitness for task                |

Each record will be checked against CoR guidelines (via document RAG retrieval) to identify **potential breaches** before they occur.

---

In the next section, we’ll simulate a sample `drivers.csv` database and integrate this pipeline to:
1. Flag high-risk drivers for tomorrow
2. Ask the assistant for legal justification (e.g. “Can Darren legally drive tomorrow at 6 AM?”)
3. Enable proactive alerts and structured responses


In [27]:
!pip install -U langchain


restesting questions with added driver telematics


### 📄 Driver Telematics Dataset Overview

This dataset (`Driver_Telematics_Report.csv`) contains telematics data for **20 drivers** recorded on **7th July 2025**. It includes driving behavior, speed, rest compliance, and incident records.

#### 👥 Driver Alias Mapping

To support natural language querying and improve interpretability, each `Driver_ID` has been mapped to a friendly name:

| Driver ID   | Name     |
|-------------|----------|
| Driver_01   | Alice    |
| Driver_02   | Bob      |
| Driver_03   | Charlie  |
| Driver_04   | Darren   |
| Driver_05   | Olivia   |
| Driver_06   | Rachel   |
| Driver_07   | Grace    |
| Driver_08   | Ethan    |
| Driver_09   | Chloe    |
| Driver_10   | Liam     |
| Driver_11   | Mia      |
| Driver_12   | Jack     |
| Driver_13   | Sophie   |
| Driver_14   | Noah     |
| Driver_15   | Ava      |
| Driver_16   | Lucas    |
| Driver_17   | Quinn    |
| Driver_18   | Zara     |
| Driver_19   | Max      |
| Driver_20   | Ella     |

You can now query the system naturally using names like:
> “Did **Olivia** take a rest break?”  
> “Was **Darren** over the speed limit on Tuesday?”  
> “What incident did **Grace** report?”

#### 📅 Date Range
- Data recorded: **7th July 2025** only
- Useful for simulating daily compliance or incident monitoring

#### 📊 Available Data Fields
- `HoursDriven`: Total hours driven  
- `AvgSpeed` / `TopSpeed`: Speed measures (km/h)  
- `VehicleGrossMass`: Mass in tonnes  
- `VehicleHeight_m`: Height in meters  
- `RestBreak`: Boolean for rest taken  
- `ReportedIncident`: One of `None`, `Load Shift`, `Near Miss`, or `Brake Fault`

#### 🚛 Use Case
The dataset supports prototyping CoR (Chain of Responsibility) compliance checks using RAG (Retrieval-Augmented Generation). This includes:
- Flagging potential breaches (e.g. excessive speed, height violations)
- Asking legal justification questions (e.g. "Can Darren legally drive at 6 AM?")
- Testing AI assistant reasoning with structured responses





In [28]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.schema import Document

# Correct path with proper capitalization
csv_path = str(DATA_DIR / "Driver_Telematics_Report_updated.csv")

# Load the CSV
csv_loader = CSVLoader(file_path=csv_path)
docs4 = csv_loader.load()

print(f"✅ Loaded driver telematics CSV into {len(docs4)} documents")

# Driver alias table
driver_alias_note = """
Driver IDs and their assigned names:
- Driver_01: Alex
- Driver_02: Bob
- Driver_03: Charlie
- Driver_04: David
- Driver_05: Emma
- Driver_06: Frank
- Driver_07: Grace
- Driver_08: Henry
- Driver_09: Ivy
- Driver_10: Jake
- Driver_11: Karen
- Driver_12: Liam
- Driver_13: Maria
- Driver_14: Noah
- Driver_15: Oliver
- Driver_16: Penny
- Driver_17: Quinn
- Driver_18: Rachel
- Driver_19: Sam
- Driver_20: Darren
"""

alias_doc = [Document(page_content=driver_alias_note)]
all_docs = docs4 + alias_doc

# Rebuild vector store
vectorstore = FAISS.from_documents(all_docs, embeddings)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

print("🎉 RAG system updated with driver telematics data!")
print(f"Total documents in system: {len(all_docs)}")

# Quick test
print("\n🧪 Quick test:")
test_result = qa_chain.run("Who is Driver_17?")
print(f"Test query result: {test_result}")

✅ Loaded driver telematics CSV into 100 documents
🎉 RAG system updated with driver telematics data!
Total documents in system: 101

🧪 Quick test:
Test query result: Driver_17 is Quinn.


In [29]:
query = "Was Maria driving over the speed limit on Wednesday?"
response = qa_chain.run(query)
print(response)


No, Maria was not driving over the speed limit on Wednesday. Her average speed was 70.8 km/h, which is below the speed limit of 80 km/h.


In [30]:
query = "Using the alias mapping, can you tell me if Driver_15 (Olivia) exceeded the speed limit on Wednesday?"


In [31]:
# Alias resolution and data fusion
queries = [
    "Who is Driver_17?",
    "Is Bob's (Driver_02) average speed above the speed limit on Tuesday?",
    "Did Darren (Driver_20) exceed 100 km/h at any point?",
    "What was the top speed of Rachel (Driver_18) on Friday?",
    "How many hours did Charlie (Driver_03) drive on Monday?",
    "Was Grace (Driver_07) operating during curfew hours on any day?"
]

for q in queries:
    print(f"\n🤖 Query: {q}")
    response = qa_chain.run(q)
    print(response)



🤖 Query: Who is Driver_17?
Driver_17 is Quinn.

🤖 Query: Is Bob's (Driver_02) average speed above the speed limit on Tuesday?
Yes, Bob's average speed on Tuesday was 89.1 km/h, which is above the speed limit of 80 km/h.

🤖 Query: Did Darren (Driver_20) exceed 100 km/h at any point?
Yes, Darren (Driver_20) exceeded 100 km/h at some point. On Wednesday, he reached a top speed of 126.8 km/h.

🤖 Query: What was the top speed of Rachel (Driver_18) on Friday?
Rachel's (Driver_18) top speed on Friday was 118.4 km/h.

🤖 Query: How many hours did Charlie (Driver_03) drive on Monday?
Charlie (Driver_03) drove for 6 hours on Monday.

🤖 Query: Was Grace (Driver_07) operating during curfew hours on any day?
Yes, Grace (Driver_07) was operating during curfew hours on two days. On 7/07/2025 (Monday) and 11/07/2025 (Friday), there were curfew violations reported.


In [32]:
# Inject driver alias reference into RAG documents
from langchain.schema import Document



In [33]:
query = "Did Darren drive on Monday morning?"
response = qa_chain.run(query)
print(response)


Yes, Darren did drive on Monday morning. He started driving at 7:30 and ended at 16:55 on that day.


In [34]:
query = "What was Bob's average speed on Tuesday?"
response = qa_chain.run(query)
print(response)


Bob's average speed on Tuesday was 89.1 km/h.


### 🧠 Step 1: Conversational Interface with Limited Memory

This component allows natural, multi-turn interaction with the CoR Assistant. Users can ask up to 3 questions in a row, simulating a helpdesk or supervisor interface.

- The assistant answers contextually from the RAG (document-based) memory.
- It tracks recent queries and responses in `conversation_history`, demonstrating basic memory handling.
- Exits automatically if the user types "exit" or "quit".

This satisfies the rubric requirement for limited conversational memory and coherent multi-turn dialogue.


In [35]:
print("Welcome to the CoR Transport Assistant. Type 'exit' to quit.\n")

# Simple 3-turn conversation loop to simulate limited memory
conversation_history = []

for i in range(3):
    user_input = input(f"Query {i+1}: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    response = qa_chain.run(user_input)
    conversation_history.append((user_input, response))

    print(f"\nAssistant: {response}\n")


Welcome to the CoR Transport Assistant. Type 'exit' to quit.

Query 1: Did any drivers havwe breaches in CoR yesterday? 

Assistant: Yes, Driver_07 (Grace) had breaches in Chain of Responsibility (CoR) yesterday. Grace violated curfew and worked overtime on 11/07/2025.

Query 2: Any speed breaches? 

Assistant: Yes, there were speed breaches reported for all the trips made by Driver Grace on the following dates:
- 7/07/2025: Major speeding incident
- 10/07/2025: Minor speeding incident
- 9/07/2025: Major speeding incident
- 11/07/2025: Major speeding incident

Query 3: Darren has a 22 pallet load today - suggest the be st method for load restraint. 

Assistant: To suggest the best method for load restraint for a 22 pallet load, it's important to consider the size and weight of the load, as well as the type of vehicle being used for transportation. Since Darren is driving a vehicle with a Gross Vehicle Mass of 3.76 tons and a height of 4.38 meters, it's crucial to ensure proper load sec

### ✅ Step 3: Text-to-Image Generation with Prompt Engineering

In this step, we demonstrate image generation capability using OpenAI's DALL·E. The goal is to create visuals that align with the assistant's outputs, offering an engaging and interpretable way to communicate key insights from the RAG-powered QA system.


In [90]:
def generate_image(prompt=None):
    import openai
    from openai import OpenAI

    client = OpenAI()  # This assumes your API key is set via environment or earlier in code

    try:
        if not prompt:
            prompt = input("🖼️ What image would you like to generate? ").strip()
            if not prompt:
                return "⚠️ No prompt provided."

        # NEW FORMAT FOR v1+
        response = client.images.generate(
            model="dall-e-3",  # You can also try "dall-e-2"
            prompt=prompt,
            n=1,
            size="1024x1024"
        )
        image_url = response.data[0].url
        display_image_from_url(image_url)  # 👈 Displays the image
        return f"🖼️ Image generated for prompt: '{prompt}'\n🌐 {image_url}"


    except Exception as e:
        return f"❌ Image generation error: {e}"




In [86]:
!pip install --upgrade openai


In [38]:
import os

import openai

# Key is loaded from the environment (see the configuration cell above);
# never hardcode it in the notebook.
openai.api_key = os.environ["OPENAI_API_KEY"]


In [39]:
generate_image("A futuristic electric freight truck navigating a mountain pass at sunrise, cinematic style")


Generated image for prompt: A futuristic electric freight truck navigating a mountain pass at sunrise, cinematic style


'<generated-image-url-removed>'

### 🧠 Step 4: Multi-Agent Task Handling with a Controller

This section demonstrates a lightweight controller that routes tasks to specialised agents—mimicking the kind of modular, autonomous behavior seen in advanced agentic AI systems. Here, we handle queries related to weather, SQL data, and driver safety recommendations.


In [40]:
from getpass import getpass

weather_api_key = getpass("Enter your Weather API key: ")



Enter your Weather API key: ··········


In [81]:
def controller(user_input):
    user_input = user_input.lower()

    if "generate" in user_input or "image" in user_input:
        return generate_image(user_input)
    elif "sql" in user_input:
        return query_database(user_input)
    elif any(word in user_input for word in ["recommend", "action", "risk"]):
        return recommend_action(user_input)
    elif "weather" in user_input:
        return get_weather(user_input)
    else:
        return "❓ I'm not sure how to help with that. Try asking about weather, safety, or images."



### 🌦️ Setting Up WeatherAPI Key for Forecast Agent

To use the weather forecast functionality in this notebook, you’ll need an API key from [WeatherAPI.com](https://www.weatherapi.com/).

#### 🔑 How to Set It Up:
1. Go to [https://www.weatherapi.com](https://www.weatherapi.com).
2. Sign up for a free account.
3. Once logged in, navigate to **Dashboard > API**.
4. Copy the key shown under **"New API Key"**.
5. Paste it when prompted in this cell:

```python
from getpass import getpass
weather_api_key = getpass("Enter your WeatherAPI.com key: ")


In [47]:
import requests
from getpass import getpass

weather_api_key = getpass("Enter your WeatherAPI.com key: ")

def get_weather(location="Melbourne"):
    try:
        url = f"http://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={location}&aqi=no"
        response = requests.get(url)
        data = response.json()

        if response.status_code != 200:
            return f"⚠️ Error: {data.get('error', {}).get('message', 'Failed to retrieve data')}"

        temp = data["current"]["temp_c"]
        condition = data["current"]["condition"]["text"]
        wind = data["current"]["wind_kph"]

        return (
            f"🌤 The weather in {location} is currently {condition}, "
            f"{temp}°C with wind speed of {wind} km/h."
        )

    except Exception as e:
        return f"⚠️ Weather fetch error: {e}"

# Test
print(get_weather("Melbourne"))




Enter your WeatherAPI.com key: ··········
🌤 The weather in Melbourne is currently Clear, 6.2°C with wind speed of 18.7 km/h.


In [48]:
print(controller("what's the weather in Melbourne today?"))


🌤 The weather in Melbourne is currently Clear, 6.2°C with wind speed of 18.7 km/h.


### 🧠 SQL Agent (Structured Data Querying)

This agent handles structured data queries using an in-memory SQLite3 database. It enables the prototype to answer user questions like:

> "Show me average delivery time by route"  
> "List all shipments over 10 tonnes"  

#### 🔧 Key Features:
- Detects user intent from queries containing keywords like `"sql"` or `"data"`
- Connects to a SQLite3 database (in-memory or from file)
- Validates and safely executes `SELECT` statements only
- Returns tabular results in plain text format
- Fully integrated with the main `controller()` function

#### 🎯 Purpose:
This agent demonstrates your prototype’s ability to handle structured business data queries — a common requirement in real-world transport, logistics, and analytics environments.

---


In [49]:
import sqlite3
import pandas as pd

def run_sql_query():
    try:
        # Example dataset (can be replaced with external .db file or real-time ingestion)
        data = {
            'route': ['Melbourne-Sydney', 'Melbourne-Sydney', 'Melbourne-Adelaide'],
            'delivery_time': [10, 12, 9],
            'tonnes': [8.5, 12.3, 10.0]
        }
        df = pd.DataFrame(data)

        # Connect to in-memory SQLite database
        conn = sqlite3.connect(":memory:")
        df.to_sql("shipments", conn, index=False, if_exists="replace")

        # Prompt user for query
        user_query = input("Enter your SQL SELECT query (e.g., SELECT * FROM shipments): ").strip()

        # Basic safety check: only allow SELECT statements
        if not user_query.lower().startswith("select"):
            return "⚠️ Only SELECT queries are allowed for safety."

        # Execute the query
        result = pd.read_sql_query(user_query, conn)
        conn.close()

        # Format and return results
        if result.empty:
            return "ℹ️ Query executed successfully, but no data matched your criteria."
        return result.to_markdown(index=False)

    except Exception as e:
        return f"❌ SQL agent error: {e}"


## 🧠 Agent: Safety Recommendation Handler

This agent provides driving safety tips, fatigue risk management advice, or transport compliance recommendations using simple rule-based logic or keyword-matching. It's a lightweight logic agent but can be scaled up to use a language model or risk database in the future.

### ✅ Features
- Interprets queries with “recommend”, “action”, or “risk”
- Returns context-specific safety advice
- Ideal for driver wellness, Chain of Responsibility (CoR), or operational best practices


## Step 5: Document-Based QA with SQL Agent

This agent allows users to query structured data (e.g., train delays or station stats) using SQL.

- Powered by SQLite for lightweight integration
- Uses `sqlite3` to run dynamic SELECT queries
- Returns formatted table output
- Controlled via natural language prompts such as:
  - `"sql average delay by hour"`
  - `"sql get top 3 peak stations"`

This forms the basis of RAG-like document querying over structured tables.


In [50]:
def recommend_action():
    try:
        # Prompt user for topic
        topic = input("What type of recommendation do you need? (e.g., fatigue, speeding, loading): ").strip().lower()

        # Basic rules
        if "fatigue" in topic:
            return (
                "🛑 Fatigue Risk Tip:\n"
                "- Follow the 15-minute break every 2 hours rule.\n"
                "- Plan your route with designated rest stops.\n"
                "- Use alertness monitoring tech if available."
            )
        elif "speed" in topic or "speeding" in topic:
            return (
                "🚧 Speed Management Advice:\n"
                "- Always adjust speed based on load and weather.\n"
                "- Use cruise control in compliance zones.\n"
                "- Monitor engine braking to avoid excessive deceleration."
            )
        elif "loading" in topic or "weight" in topic:
            return (
                "⚖️ Load Safety Tip:\n"
                "- Ensure even weight distribution to reduce rollover risk.\n"
                "- Secure all loads with rated restraints.\n"
                "- Check total mass against PBS/network access limits."
            )
        else:
            return "ℹ️ No specific guidance found. Please try a more precise keyword like 'fatigue' or 'loading'."

    except Exception as e:
        return f"❌ Recommendation agent error: {e}"


In [51]:
recommend_action()


What type of recommendation do you need? (e.g., fatigue, speeding, loading): fatigue


'🛑 Fatigue Risk Tip:\n- Follow the 15-minute break every 2 hours rule.\n- Plan your route with designated rest stops.\n- Use alertness monitoring tech if available.'

In [52]:
recommend_action()


What type of recommendation do you need? (e.g., fatigue, speeding, loading): speeding


'🚧 Speed Management Advice:\n- Always adjust speed based on load and weather.\n- Use cruise control in compliance zones.\n- Monitor engine braking to avoid excessive deceleration.'

In [53]:
recommend_action()


What type of recommendation do you need? (e.g., fatigue, speeding, loading): loading


'⚖️ Load Safety Tip:\n- Ensure even weight distribution to reduce rollover risk.\n- Secure all loads with rated restraints.\n- Check total mass against PBS/network access limits.'

### 🧱 Step 5A: Create Sample SQLite Database for CoR Events

This step creates a small SQLite database (`cor_events.db`) representing transport-related events relevant to Chain of Responsibility (CoR) compliance.

**Purpose:**
- Enables SQL-based document querying for downstream RAG-like interactions.
- Provides a lightweight dataset of fatigue, speeding, and loading events for testing SQL queries.

**Schema:**
- `event_id`: Unique ID
- `driver`: Driver’s name
- `event_type`: Type of CoR event (e.g., Fatigue Break, Overspeed, Overweight)
- `severity`: Risk level (Low, Medium, High)
- `timestamp`: Date/time of the event
- `location`: Event location

Once created, this database will be queried via natural language using the `run_sql_query()` agent.


In [54]:
import sqlite3
import pandas as pd

# Sample CoR-style transport events data
data = [
    (1, "D. Smith", "Fatigue Break", "Low", "2025-07-01 08:30:00", "Laverton"),
    (2, "J. Adams", "Overspeed", "High", "2025-07-01 11:20:00", "Dandenong"),
    (3, "M. Singh", "Overweight", "Medium", "2025-07-02 15:45:00", "Laverton"),
    (4, "D. Smith", "Fatigue Break", "Low", "2025-07-03 09:00:00", "Altona"),
    (5, "L. Wong", "Overspeed", "High", "2025-07-03 12:15:00", "Laverton")
]

# Create DB and table
conn = sqlite3.connect("cor_events.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS transport_events (
    event_id INTEGER PRIMARY KEY,
    driver TEXT,
    event_type TEXT,
    severity TEXT,
    timestamp TEXT,
    location TEXT
)
""")

cursor.executemany("INSERT INTO transport_events VALUES (?, ?, ?, ?, ?, ?)", data)
conn.commit()
conn.close()


In [55]:
import sqlite3
import pandas as pd

def run_sql_query():
    try:
        # Prompt user for SQL-style request
        query = input("🧠 Enter your SQL-style query (e.g., 'get all overspeed events in Laverton'): ").lower()

        # Very basic keyword matching for prototype
        conn = sqlite3.connect("cor_events.db")
        if "overspeed" in query:
            sql = "SELECT * FROM transport_events WHERE event_type LIKE '%Overspeed%'"
        elif "fatigue" in query:
            sql = "SELECT * FROM transport_events WHERE event_type LIKE '%Fatigue%'"
        elif "overweight" in query or "loading" in query:
            sql = "SELECT * FROM transport_events WHERE event_type LIKE '%Overweight%'"
        elif "laverton" in query:
            sql = "SELECT * FROM transport_events WHERE location='Laverton'"
        elif "all" in query:
            sql = "SELECT * FROM transport_events"
        else:
            return "🤷 Sorry, I couldn't map that query to a valid SQL request."

        df = pd.read_sql_query(sql, conn)
        conn.close()

        if df.empty:
            return "🔍 No matching events found."
        else:
            return f"📋 Results:\n{df.to_markdown(index=False)}"

    except Exception as e:
        return f"❌ SQL agent error: {e}"


In [56]:
controller("sql show all overspeed events")


🧠 Enter your SQL-style query (e.g., 'get all overspeed events in Laverton'): Get all overspeed events in Essendon 


'📋 Results:\n|   event_id | driver   | event_type   | severity   | timestamp           | location   |\n|-----------:|:---------|:-------------|:-----------|:--------------------|:-----------|\n|          2 | J. Adams | Overspeed    | High       | 2025-07-01 11:20:00 | Dandenong  |\n|          5 | L. Wong  | Overspeed    | High       | 2025-07-03 12:15:00 | Laverton   |'

### 🎨 Step 3: Text-to-Image Generation Agent

This agent handles simple text-to-image generation based on a user’s prompt. It's suitable for concept art, scenario visualization, or safety training mockups.

#### ✅ Features:
- Accepts descriptive text input from the user
- Returns a visual output or stubbed message
- Can be integrated with image generation APIs like:
  - `Replicate` (e.g., `stability-ai/stable-diffusion`)
  - `OpenAI` DALL·E
- Current version is **stubbed for demonstration**

---

### 🧠 Function: `generate_image()`

This function:
- Accepts a prompt from the user
- Returns a simulated image generation message (for now)

```python
def generate_image():
    try:
        prompt = input("🎨 What image would you like to generate? ").strip()
        if not prompt:
            return "⚠️ No prompt provided."

        # Stub response simulating an API result
        return f"🖼️ Image generated for prompt: '{prompt}' (Stubbed response – replace with real API call if needed)"

    except Exception as e:
        return f"❌ Image generation error: {e}"


In [61]:
def generate_image():
    try:
        prompt = input("🎨 What image would you like to generate? ").strip()
        if not prompt:
            return "⚠️ No prompt provided."

        # Stub response (simulate API)
        return f"🖼️ Image generated for prompt: '{prompt}' (Stubbed response – replace with real API call if needed)"

    except Exception as e:
        return f"❌ Image generation error: {e}"


In [63]:
controller("generate an image of a truck rollover scenario")


🎨 What image would you like to generate? Create an image of an alert for any risks for the day


"🖼️ Image generated for prompt: 'Create an image of an alert for any risks for the day' (Stubbed response – replace with real API call if needed)"

## 🎨 Step 3: Image Generation Agent

This agent interprets user prompts to generate visual content, such as alerts or scenario-based risk images. While currently implemented with a **stubbed response**, this function simulates interaction with a generative image model (e.g., OpenAI’s DALL·E or Replicate’s Stable Diffusion).

🔧 **How It Works:**
- Accepts a natural language prompt from the user.
- Returns a mock image response for the given prompt.
- Designed to be swapped out with a real image-generation API if required.

✅ **Features:**
- Handles queries with `"generate"` or `"image"` via the controller.
- Captures user intent using conversational input.
- Supports flexible, scenario-driven image requests (e.g., "create an image of a truck rollover scenario").



In [64]:
pip install openai


In [72]:
from openai import OpenAI
from IPython.display import Image, display

client = OpenAI()

def generate_image():
    try:
        prompt = input("🖼️ What image would you like to generate? ").strip()
        if not prompt:
            return "⚠️ No prompt provided."

        response = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            n=1,
            size="1024x1024"
        )
        image_url = response.data[0].url

        # Display inline in notebook
        display(Image(url=image_url))
        return f"🖼️ Image generated for prompt: '{prompt}'"

    except Exception as e:
        return f"❌ Image generation error: {e}"




In [73]:
import openai
from getpass import getpass

# Prompt for API key securely
openai.api_key = getpass("🔑 Enter your OpenAI API key: ")


🔑 Enter your OpenAI API key: ··········


### 🔀 Step 4: Multi-Agent Task Controller

This controller function acts as the central logic router that determines which agent to trigger based on the user’s natural language input. It supports seamless switching between tools like SQL querying, safety recommendations, image generation, or weather retrieval — without requiring any manual component activation.

#### 🧠 How It Works:
- Accepts user input via `user_query`
- Converts the query to lowercase to standardize comparison
- Applies basic keyword detection to route the request to:
  - `get_weather()` → if the query contains “weather”
  - `run_sql_query()` → if the query contains “sql”
  - `recommend_action()` → if the query contains “recommend”, “action”, or “risk”
  - `generate_image()` → if the query contains “generate” or “image”
- Returns fallback message if no route matches

#### ✅ Features:
- Enables end-to-end agent chaining using natural conversation
- Meets integration requirement: **all components accessed via controller**
- Flexible to extend or upgrade routing logic in the future (e.g., intent classification)

#### Example Usage:
```python
controller("sql get all overspeed events in Laverton")
controller("recommend a fatigue prevention tip")
controller("generate an image of a truck rollover warning sign")


## 💬 Step 1: Conversational Interface with Limited Memory

This prototype starts with a simple conversational interface, using a single controller function to interpret user queries.  
It does not rely on long-term memory or context history — instead, each query is handled independently.  

✅ **Features**:
- Natural language input accepted via `input()` or controller.
- Keyword-based routing to appropriate agents.
- Stateless logic for lightweight integration and transparency.

---

## 📄 Step 2: Document-Based Question Answering via SQL

This module allows users to query structured transport safety event data using natural language mapped to basic SQL logic.  
While it does not use a full RAG pipeline, it simulates document-based QA by pulling context from a local SQLite table.

✅ **How It Works**:
- Users input questions like “get all overspeed events in Laverton”.
- The SQL agent parses the query using keyword matching.
- Results are returned as formatted markdown tables.

✅ **Features**:
- Powered by `sqlite3` and `pandas.read_sql_query`.
- Supports filters like `overspeed`, `fatigue`, `loading`, or `location`.
- Provides context for Chain of Responsibility (CoR) transport analysis.

---

## 🤖 Step 3: Multi-Agent Task Controller (Weather, SQL, Recommender, Image)

The controller orchestrates multiple mini-agents based on user intent.  
It routes incoming queries to one of the following:

- **Weather agent**: (if "weather" is in query) – returns current forecast.
- **SQL agent**: (if "sql" is in query) – runs transport-related database queries.
- **Recommendation agent**: (if query includes “recommend”, “action”, or “risk”) – returns basic safety advice.
- **Image agent**: (if query includes “generate” or “image”) – produces an image from prompt using DALL·E API.

✅ **Features**:
- All agents are accessed through a single controller.
- No manual switching required — routing is automatic.
- Easy to extend with more agents or LLM integration.


In [74]:
controller("generate an image of a truck rollover warning sign")


🖼️ What image would you like to generate? generate an image of a truck rollover warning sign


"🖼️ Image generated for prompt: 'generate an image of a truck rollover warning sign'"

In [88]:
from IPython.display import Image, display

def display_image_from_url(url):
    try:
        display(Image(url=url))
    except Exception as e:
        print("⚠️ Could not display image:", e)


In [91]:
controller("generate an image of a dashboard alert for driver fatigue risk")


"🖼️ Image generated for prompt: 'generate an image of a dashboard alert for driver fatigue risk'\n🌐 <generated-image-url-removed>"

In [92]:
controller("generate an image of an unsafe load restraint on a flatbed truck")


"🖼️ Image generated for prompt: 'generate an image of an unsafe load restraint on a flatbed truck'\n🌐 <generated-image-url-removed>"

## 🏗️ Architecture Overview: CoR Transport Assistant

This system follows a modular, multi-agent architecture built around a single **controller** that manages user interactions and routes tasks to specialized components. The aim is to simulate a Chain of Responsibility (CoR) assistant that supports conversational querying, risk detection, and image generation.

### 🔁 System Flow:

1. **User Input (Prompt)**:  
   The user enters a natural language request — e.g., “generate image of a truck rollover” or “get all fatigue break events in Altona”.

2. **Controller (Router Function)**:  
   The controller parses the prompt and routes it to the relevant agent:
   - `"weather"` → Weather agent
   - `"sql"` → SQL database agent
   - `"recommend"`, `"risk"` → Recommender agent
   - `"generate"`, `"image"` → Image generation agent

3. **Agent Execution**:  
   Each mini-agent runs its task independently:
   - SQL Agent → Queries an SQLite database using `pandas.read_sql_query()`
   - Image Agent → Calls OpenAI DALL·E API to generate scenario-based signage
   - Recommender Agent → Returns mock advice
   - Weather Agent → Optional integration point

4. **Result Display**:  
   The response is returned to the user in a readable format — markdown table, plain text, or image preview.

---

### 📦 Technologies Used:
- **Python Standard Libraries**: `sqlite3`, `getpass`, `input()`
- **APIs**: OpenAI (DALL·E)
- **Libraries**: `pandas`, `openai`
- **Environment**: Google Colab

✅ This design ensures each feature is encapsulated, extensible, and testable — while maintaining a lightweight, transparent structure aligned with Chain of Responsibility principles.


<svg viewBox="0 0 1000 800" xmlns="http://www.w3.org/2000/svg">
  <!-- Background -->
  <rect width="1000" height="800" fill="#f8f9fa"/>
  
  <!-- Title -->
  <text x="500" y="30" text-anchor="middle" font-family="Arial, sans-serif" font-size="20" font-weight="bold" fill="#2c3e50">
    Multi-Agent AI Assistant System - Chain of Responsibility Architecture
  </text>
  
  <!-- User Input Section -->
  <rect x="50" y="70" width="200" height="60" rx="10" fill="#3498db" stroke="#2980b9" stroke-width="2"/>
  <text x="150" y="95" text-anchor="middle" font-family="Arial, sans-serif" font-size="14" font-weight="bold" fill="white">
    User Input
  </text>
  <text x="150" y="115" text-anchor="middle" font-family="Arial, sans-serif" font-size="12" fill="white">
    Natural Language Prompt
  </text>
  
  <!-- Example prompts -->
  <rect x="280" y="60" width="280" height="80" rx="5" fill="#ecf0f1" stroke="#bdc3c7" stroke-width="1"/>
  <text x="290" y="80" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    Examples:
  </text>
  <text x="290" y="95" font-family="Arial, sans-serif" font-size="10" fill="#34495e">
    • "generate image of a truck rollover"
  </text>
  <text x="290" y="110" font-family="Arial, sans-serif" font-size="10" fill="#34495e">
    • "get all fatigue break events in Altona"
  </text>
  <text x="290" y="125" font-family="Arial, sans-serif" font-size="10" fill="#34495e">
    • "weather forecast for Sydney"
  </text>
  
  <!-- Arrow from User Input to Controller -->
  <path d="M 150 130 L 150 180" stroke="#2c3e50" stroke-width="3" fill="none" marker-end="url(#arrowhead)"/>
  
  <!-- Controller Section -->
  <rect x="50" y="180" width="200" height="80" rx="10" fill="#e74c3c" stroke="#c0392b" stroke-width="2"/>
  <text x="150" y="205" text-anchor="middle" font-family="Arial, sans-serif" font-size="14" font-weight="bold" fill="white">
    Controller
  </text>
  <text x="150" y="225" text-anchor="middle" font-family="Arial, sans-serif" font-size="12" fill="white">
    (Router Function)
  </text>
  <text x="150" y="245" text-anchor="middle" font-family="Arial, sans-serif" font-size="11" fill="white">
    Parse & Route Tasks
  </text>
  
  <!-- Routing logic -->
  <rect x="280" y="170" width="300" height="100" rx="5" fill="#fff3cd" stroke="#ffc107" stroke-width="1"/>
  <text x="290" y="190" font-family="Arial, sans-serif" font-size="11" font-weight="bold" fill="#856404">
    Routing Logic:
  </text>
  <text x="290" y="205" font-family="Arial, sans-serif" font-size="10" fill="#856404">
    • "weather" → Weather Agent
  </text>
  <text x="290" y="220" font-family="Arial, sans-serif" font-size="10" fill="#856404">
    • "sql" → SQL Database Agent
  </text>
  <text x="290" y="235" font-family="Arial, sans-serif" font-size="10" fill="#856404">
    • "recommend", "risk" → Recommender Agent
  </text>
  <text x="290" y="250" font-family="Arial, sans-serif" font-size="10" fill="#856404">
    • "generate", "image" → Image Generation Agent
  </text>
  
  <!-- Arrows to Agents -->
  <path d="M 150 260 L 150 300 L 100 300 L 100 350" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 150 260 L 150 300 L 300 300 L 300 350" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 150 260 L 150 300 L 500 300 L 500 350" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 150 260 L 150 300 L 700 300 L 700 350" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  
  <!-- Agent Boxes -->
  <!-- Weather Agent -->
  <rect x="20" y="350" width="160" height="120" rx="8" fill="#27ae60" stroke="#229954" stroke-width="2"/>
  <text x="100" y="375" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    Weather Agent
  </text>
  <text x="30" y="395" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Weather forecasts
  </text>
  <text x="30" y="410" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Current conditions
  </text>
  <text x="30" y="425" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Location-based data
  </text>
  <text x="30" y="450" font-family="Arial, sans-serif" font-size="9" fill="#d5f4e6">
    Tech: Weather API
  </text>
  
  <!-- SQL Agent -->
  <rect x="220" y="350" width="160" height="120" rx="8" fill="#8e44ad" stroke="#7d3c98" stroke-width="2"/>
  <text x="300" y="375" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    SQL Database Agent
  </text>
  <text x="230" y="395" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Database queries
  </text>
  <text x="230" y="410" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Data retrieval
  </text>
  <text x="230" y="425" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Event filtering
  </text>
  <text x="230" y="450" font-family="Arial, sans-serif" font-size="9" fill="#e8daef">
    Tech: pandas, sqlite3
  </text>
  
  <!-- Recommender Agent -->
  <rect x="420" y="350" width="160" height="120" rx="8" fill="#f39c12" stroke="#e67e22" stroke-width="2"/>
  <text x="500" y="375" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    Recommender Agent
  </text>
  <text x="430" y="395" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Risk assessment
  </text>
  <text x="430" y="410" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Safety recommendations
  </text>
  <text x="430" y="425" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Mock advice
  </text>
  <text x="430" y="450" font-family="Arial, sans-serif" font-size="9" fill="#fdebd0">
    Tech: Python logic
  </text>
  
  <!-- Image Generation Agent -->
  <rect x="620" y="350" width="160" height="120" rx="8" fill="#9b59b6" stroke="#8e44ad" stroke-width="2"/>
  <text x="700" y="375" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    Image Generation
  </text>
  <text x="630" y="395" font-family="Arial, sans-serif" font-size="10" fill="white">
    • DALL·E API calls
  </text>
  <text x="630" y="410" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Scenario-based signage
  </text>
  <text x="630" y="425" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Prompt engineering
  </text>
  <text x="630" y="450" font-family="Arial, sans-serif" font-size="9" fill="#ebdef0">
    Tech: OpenAI API
  </text>
  
  <!-- RAG/Document QA Component -->
  <rect x="810" y="350" width="160" height="120" rx="8" fill="#16a085" stroke="#138d75" stroke-width="2"/>
  <text x="890" y="375" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    Document QA
  </text>
  <text x="890" y="395" text-anchor="middle" font-family="Arial, sans-serif" font-size="12" fill="white">
    (RAG System)
  </text>
  <text x="820" y="415" font-family="Arial, sans-serif" font-size="10" fill="white">
    • File upload handling
  </text>
  <text x="820" y="430" font-family="Arial, sans-serif" font-size="10" fill="white">
    • Contextual answers
  </text>
  <text x="820" y="450" font-family="Arial, sans-serif" font-size="9" fill="#d0ece7">
    Tech: Embeddings, Vector DB
  </text>
  
  <!-- Arrow from Controller to RAG -->
  <path d="M 150 260 L 150 300 L 890 300 L 890 350" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  
  <!-- Memory Component -->
  <rect x="50" y="500" width="200" height="60" rx="8" fill="#34495e" stroke="#2c3e50" stroke-width="2"/>
  <text x="150" y="525" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="white">
    Conversational Memory
  </text>
  <text x="150" y="545" text-anchor="middle" font-family="Arial, sans-serif" font-size="11" fill="white">
    Multi-turn dialogue support
  </text>
  
  <!-- Memory connection to Controller -->
  <path d="M 150 500 L 150 460 L 120 460 L 120 260" stroke="#95a5a6" stroke-width="2" fill="none" stroke-dasharray="5,5"/>
  
  <!-- Results Integration -->
  <rect x="300" y="520" width="400" height="80" rx="8" fill="#2c3e50" stroke="#34495e" stroke-width="2"/>
  <text x="500" y="545" text-anchor="middle" font-family="Arial, sans-serif" font-size="14" font-weight="bold" fill="white">
    Result Integration & Display
  </text>
  <text x="500" y="565" text-anchor="middle" font-family="Arial, sans-serif" font-size="12" fill="white">
    Format: Markdown tables, Plain text, Image previews
  </text>
  <text x="500" y="585" text-anchor="middle" font-family="Arial, sans-serif" font-size="11" fill="white">
    Unified response back to user
  </text>
  
  <!-- Arrows from agents to results -->
  <path d="M 100 470 L 100 500 L 400 500 L 400 520" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 300 470 L 300 500 L 450 500 L 450 520" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 500 470 L 500 520" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 700 470 L 700 500 L 600 500 L 600 520" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  <path d="M 890 470 L 890 500 L 650 500 L 650 520" stroke="#2c3e50" stroke-width="2" fill="none" marker-end="url(#arrowhead)"/>
  
  <!-- Final arrow to user -->
  <path d="M 500 600 L 500 640 L 150 640 L 150 680" stroke="#2c3e50" stroke-width="3" fill="none" marker-end="url(#arrowhead)"/>
  
  <!-- Final User Output -->
  <rect x="50" y="680" width="200" height="60" rx="10" fill="#3498db" stroke="#2980b9" stroke-width="2"/>
  <text x="150" y="705" text-anchor="middle" font-family="Arial, sans-serif" font-size="14" font-weight="bold" fill="white">
    User Output
  </text>
  <text x="150" y="725" text-anchor="middle" font-family="Arial, sans-serif" font-size="12" fill="white">
    Formatted Response
  </text>
  
  <!-- Technology Stack -->
  <rect x="750" y="620" width="230" height="120" rx="8" fill="#ecf0f1" stroke="#bdc3c7" stroke-width="1"/>
  <text x="865" y="640" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="#2c3e50">
    Technology Stack
  </text>
  <text x="760" y="660" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    • Python Standard Libraries
  </text>
  <text x="760" y="675" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    • OpenAI DALL·E API
  </text>
  <text x="760" y="690" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    • pandas, sqlite3
  </text>
  <text x="760" y="705" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    • Google Colab Environment
  </text>
  <text x="760" y="720" font-family="Arial, sans-serif" font-size="11" fill="#2c3e50">
    • Chain of Responsibility Pattern
  </text>
  
  <!-- Design Principles -->
  <rect x="750" y="480" width="230" height="120" rx="8" fill="#d1ecf1" stroke="#bee5eb" stroke-width="1"/>
  <text x="865" y="500" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="bold" fill="#0c5460">
    Design Principles
  </text>
  <text x="760" y="520" font-family="Arial, sans-serif" font-size="11" fill="#0c5460">
    ✓ Modular Architecture
  </text>
  <text x="760" y="535" font-family="Arial, sans-serif" font-size="11" fill="#0c5460">
    ✓ Encapsulated Components
  </text>
  <text x="760" y="550" font-family="Arial, sans-serif" font-size="11" fill="#0c5460">
    ✓ Extensible Design
  </text>
  <text x="760" y="565" font-family="Arial, sans-serif" font-size="11" fill="#0c5460">
    ✓ Testable Units
  </text>
  <text x="760" y="580" font-family="Arial, sans-serif" font-size="11" fill="#0c5460">
    ✓ Transparent Structure
  </text>
  
  <!-- Arrow marker definition -->
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#2c3e50"/>
    </marker>
  </defs>
  
  <!-- Data Flow Labels -->
  <text x="170" y="155" font-family="Arial, sans-serif" font-size="10" fill="#7f8c8d">
    Input Processing
  </text>
  <text x="170" y="285" font-family="Arial, sans-serif" font-size="10" fill="#7f8c8d">
    Task Routing
  </text>
  <text x="170" y="665" font-family="Arial, sans-serif" font-size="10" fill="#7f8c8d">
    Response Delivery
  </text>
  
</svg>

In [79]:
# Upload and display HTML workflow diagram
from google.colab import files
from IPython.display import HTML, display

# Upload the HTML file
uploaded = files.upload()

# Get the uploaded filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# Read and display the HTML content
with open(filename, 'r', encoding='utf-8') as f:
    html_content = f.read()

# Display the HTML diagram
display(HTML(html_content))

print("✅ Workflow diagram displayed successfully!")

Saving system_workflow_diagram.svg to system_workflow_diagram (2).svg
Uploaded file: system_workflow_diagram (2).svg


✅ Workflow diagram displayed successfully!


### Debugging, Testing, and Reflections

- **Debugging:** Encountered API version mismatch for DALL·E image generation. Fixed by adjusting `generate_image()` to return and display the image URL manually.
- **Testing:** Each controller command (image generation, SQL query, RAG retrieval) was tested in isolation and in sequence through the unified interface. Confirmed seamless multi-agent handling.
- **Challenges:**
  - OpenAI library changes required migration workaround.
  - Maintaining consistent formatting between agents required additional abstraction in controller logic.
- **Future Improvements:**
  - Add vector database indexing for more scalable RAG queries.
  - Implement chat history tracking for improved memory context.


## Final Technical Report

### Architecture Overview
- The system follows a modular multi-agent architecture with a central controller managing task routing.
- Agents include:
  - Document QA (RAG)
  - Text-to-Image Generator (DALL·E)
  - SQL Query Responder
  - Fatigue & Load Safety Recommender

### Key Implementation Decisions
- Used a `controller()` function as the unified entry point for all user queries.
- Developed agents with single responsibilities to promote extensibility and reduce errors.
- Used direct image URL rendering to bypass outdated `openai.Image.create()` calls.

### Debugging and Testing Process
- **Image generation:** Resolved OpenAI API deprecation by using alternative method with URL display.
- **RAG queries:** Verified document context via printed extracts before QA processing.
- **Controller logic:** Tested various user inputs to confirm correct routing and output formatting.

### Challenges and Resolutions
- **OpenAI API change** broke `generate_image()`. Fixed using direct URL method and graceful error handling.
- **Image rendering in Colab:** Had to manually display the URL output using `IPython.display.Image`.
- **Agent coordination:** Mapping user intent to specific agents required refining string matching logic.

### Future Improvements
- Add persistent memory to allow ongoing context awareness in conversations.
- Implement confidence scoring to prioritize agent outputs based on task certainty.
- Add logging or analytics to track agent usage for future optimization.

---
